In [ ]:
%pip install rdkit pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 56.9 MB/s eta 0:00:00


In [ ]:
import rdkit
import pandas as pd
import numpy as np
import sklearn
from rdkit import Chem

print("RDKit:", rdkit.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

# Basic molecular parsing test
mol = Chem.MolFromSmiles("CCO")

print("Molecule parsed:", mol is not None)
print("Atom count:", mol.GetNumAtoms() if mol else None)
print("Bond count:", mol.GetNumBonds() if mol else None)

RDKit: 2026.09.1
Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1
Molecule parsed: True
Atom count: 3
Bond count: 2


In [ ]:
url = (
    "https://deepchemdata.s3-us-west-1.amazonaws.com/"
    "datasets/tox21.csv.gz"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).head(15))

Dataset shape: (7831, 14)

Column names:
['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']

First five rows:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O



Missing values per column:


,0
SR-MMP,2021
NR-Aromatase,2010
SR-ARE,1999
NR-ER,1638
NR-PPAR-gamma,1381
SR-HSE,1364
NR-AhR,1282
NR-AR-LBD,1073
SR-p53,1057
NR-ER-LBD,876


In [ ]:
# Inspect the SMILES column
assert "smiles" in df.columns, (
    "Expected a 'smiles' column. Inspect df.columns."
)

df["smiles"] = df["smiles"].astype("string").str.strip()

print("Total records:", len(df))
print("Missing SMILES:", df["smiles"].isna().sum())
print("Duplicate SMILES strings:", df["smiles"].duplicated().sum())

display(df[["smiles"]].head(10))

Total records: 7831
Missing SMILES: 0
Duplicate SMILES strings: 0


,smiles
0,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,CCN1C(=O)NC(c2ccccc2)C1=O
2,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,CC(O)(P(=O)(O)O)P(=O)(O)O
5,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C
6,O=S(=O)(Cl)c1ccccc1
7,O=C(O)Cc1cc(I)c(Oc2ccc(O)c(I)c2)c(I)c1
8,OC[C@H](O)[C@@H](O)[C@H](O)CO
9,CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2]


In [ ]:
def check_smiles(smiles):
    if pd.isna(smiles) or not smiles:
        return False

    mol = Chem.MolFromSmiles(str(smiles))
    return mol is not None


df["rdkit_valid"] = df["smiles"].apply(check_smiles)

print("Total:", len(df))
print("RDKit-valid:", int(df["rdkit_valid"].sum()))
print("Failed parsing/sanitization:", int((~df["rdkit_valid"]).sum()))
print(
    "Validity rate:",
    round(100 * df["rdkit_valid"].mean(), 2), "%"
)

[11:44:01] WARNING: not removing hydrogen atom without neighbors
[11:44:01] Explicit valence for atom # 8 Al, 6, is greater than permitted
[11:44:01] Explicit valence for atom # 3 Al, 6, is greater than permitted
[11:44:01] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:44:01] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:44:02] Explicit valence for atom # 9 Al, 6, is greater than permitted
[11:44:02] Explicit valence for atom # 5 Al, 6, is greater than permitted
[11:44:02] Explicit valence for atom # 16 Al, 6, is greater than permitted


Total: 7831
RDKit-valid: 7823
Failed parsing/sanitization: 8
Validity rate: 99.9 %


[11:44:02] Explicit valence for atom # 20 Al, 6, is greater than permitted


In [ ]:
from rdkit import Chem
from rdkit import RDLogger

# Suppress repeated RDKit warnings during this diagnostic only.
# The failure details will still be recorded below.
RDLogger.DisableLog("rdApp.warning")

def parse_molecule(smiles):
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None

df["mol"] = df["smiles"].apply(parse_molecule)

invalid_mask = df["mol"].isna()

invalid_rows = df.loc[
    invalid_mask, ["mol_id", "smiles"]
].copy()

print("Failed molecules:", len(invalid_rows))
display(invalid_rows.to_string(index=False))

# Restore warnings for subsequent notebook cells.
RDLogger.EnableLog("rdApp.warning")

[11:49:45] Explicit valence for atom # 8 Al, 6, is greater than permitted
[11:49:46] Explicit valence for atom # 3 Al, 6, is greater than permitted
[11:49:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:49:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[11:49:46] Explicit valence for atom # 9 Al, 6, is greater than permitted
[11:49:46] Explicit valence for atom # 5 Al, 6, is greater than permitted
[11:49:47] Explicit valence for atom # 16 Al, 6, is greater than permitted
[11:49:47] Explicit valence for atom # 20 Al, 6, is greater than permitted


Failed molecules: 8


'  mol_id                                                                                                                                                                                                                                                                                                                             smiles\nTOX31563                                                                                                                                                                                                                                                                                                     NC(=O)NC1N=C(O[AlH3](O)O)NC1=O\nTOX24724                                                                                                                                                                                                                                                                                                               O=CO[AlH3](OC=O)OC=O\n

In [ ]:
def canonicalize(mol):
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

df["canonical_smiles"] = df["mol"].apply(canonicalize)

valid_df = df.loc[~invalid_mask].copy()

canonical_duplicate_mask = valid_df["canonical_smiles"].duplicated(
    keep=False
)

canonical_duplicates = valid_df.loc[
    canonical_duplicate_mask,
    ["mol_id", "smiles", "canonical_smiles"]
].sort_values("canonical_smiles")

print("RDKit-valid molecules:", len(valid_df))
print("Unique canonical structures:", valid_df["canonical_smiles"].nunique())
print("Rows belonging to canonical duplicate groups:", len(canonical_duplicates))

display(canonical_duplicates.head(30))

RDKit-valid molecules: 7823
Unique canonical structures: 7823
Rows belonging to canonical duplicate groups: 0


,mol_id,smiles,canonical_smiles


In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from pathlib import Path

OUTPUT_DIR = Path("/content/dual_axis_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Preserve failed source records for auditing
invalid_rows.to_csv(
    OUTPUT_DIR / "source_invalid_audit.csv",
    index=False
)

# Use only successfully parsed and sanitized molecules
clean_df = valid_df.copy()

def get_scaffold_key(mol, canonical_smiles):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    if scaffold is None or scaffold.GetNumAtoms() == 0:
        # Acyclic molecules have no conventional Murcko scaffold.
        # Use exact canonical structure as a documented fallback.
        return "ACYCLIC::" + canonical_smiles

    return "SCAFFOLD::" + Chem.MolToSmiles(
        scaffold, canonical=True
    )

clean_df["scaffold_key"] = clean_df.apply(
    lambda row: get_scaffold_key(
        row["mol"], row["canonical_smiles"]
    ),
    axis=1
)

print("Clean molecules:", len(clean_df))
print("Unique scaffold groups:", clean_df["scaffold_key"].nunique())
print(
    "Acyclic molecules:",
    clean_df["scaffold_key"].str.startswith("ACYCLIC::").sum()
)

display(
    clean_df[
        ["mol_id", "smiles", "canonical_smiles", "scaffold_key"]
    ].head()
)

Clean molecules: 7823
Unique scaffold groups: 4178
Acyclic molecules: 1775


,mol_id,smiles,canonical_smiles,scaffold_key
0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,SCAFFOLD::c1ccc2scnc2c1
1,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,CCN1C(=O)NC(c2ccccc2)C1=O,SCAFFOLD::O=C1NC(=O)C(c2ccccc2)N1
2,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,SCAFFOLD::C1=C2CC[C@@H]3[C@H](CCC4CCC[C@H]43)[...
3,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,SCAFFOLD::c1ccccc1
4,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,CC(O)(P(=O)(O)O)P(=O)(O)O,ACYCLIC::CC(O)(P(=O)(O)O)P(=O)(O)O


In [ ]:
SPLIT_SEED = 42

split_names = ["id_train", "id_val", "id_test", "ood_valid"]
split_proportions = {
    "id_train": 0.70,
    "id_val": 0.10,
    "id_test": 0.10,
    "ood_valid": 0.10,
}

# Count molecules in each scaffold group
group_sizes = (
    clean_df.groupby("scaffold_key")
    .size()
    .to_dict()
)

# Deterministic ordering: largest groups first, randomizing equal-size ties
rng = np.random.default_rng(SPLIT_SEED)
group_keys = list(group_sizes.keys())
tie_breakers = rng.random(len(group_keys))

groups = pd.DataFrame({
    "scaffold_key": group_keys,
    "size": [group_sizes[k] for k in group_keys],
    "tie": tie_breakers,
}).sort_values(
    ["size", "tie"],
    ascending=[False, True]
)

total = len(clean_df)
targets = {
    name: total * split_proportions[name]
    for name in split_names
}

assigned_counts = {name: 0 for name in split_names}
group_assignment = {}

# Greedily assign each entire group to the split that best
# matches the target proportions.
for row in groups.itertuples(index=False):
    key, size = row.scaffold_key, int(row.size)

    best_split = min(
        split_names,
        key=lambda name: sum(
            (
                (
                    assigned_counts[s]
                    + (size if s == name else 0)
                    - targets[s]
                ) / max(targets[s], 1)
            ) ** 2
            for s in split_names
        )
    )

    group_assignment[key] = best_split
    assigned_counts[best_split] += size

clean_df["split"] = clean_df["scaffold_key"].map(
    group_assignment
)

print("Split counts:")
display(
    clean_df["split"]
    .value_counts()
    .reindex(split_names, fill_value=0)
    .rename("molecule_count")
    .to_frame()
)

print("Split proportions:")
display(
    (
        clean_df["split"]
        .value_counts(normalize=True)
        .reindex(split_names, fill_value=0) * 100
    ).rename("percentage").to_frame()
)

Split counts:


,molecule_count
split,
id_train,5477
id_val,782
id_test,782
ood_valid,782


Split proportions:


,percentage
split,
id_train,70.011505
id_val,9.996165
id_test,9.996165
ood_valid,9.996165


In [ ]:
scaffolds_by_split = {
    name: set(
        clean_df.loc[
            clean_df["split"] == name, "scaffold_key"
        ]
    )
    for name in split_names
}

for i, first in enumerate(split_names):
    for second in split_names[i + 1:]:
        overlap = scaffolds_by_split[first] & scaffolds_by_split[second]
        print(f"{first} vs {second}: {len(overlap)} shared scaffold keys")
        assert len(overlap) == 0, (
            f"Scaffold leakage detected between {first} and {second}"
        )

print("Scaffold group isolation passed.")

id_train vs id_val: 0 shared scaffold keys
id_train vs id_test: 0 shared scaffold keys
id_train vs ood_valid: 0 shared scaffold keys
id_val vs id_test: 0 shared scaffold keys
id_val vs ood_valid: 0 shared scaffold keys
id_test vs ood_valid: 0 shared scaffold keys
Scaffold group isolation passed.


In [ ]:
# Export separate molecule tables for downstream team members.
for split_name in split_names:
    part = clean_df.loc[clean_df["split"] == split_name].copy()

    # Remove RDKit Mol objects; they are not directly CSV-serializable.
    part.drop(columns=["mol"], errors="ignore").to_csv(
        OUTPUT_DIR / f"{split_name}.csv",
        index=False
    )

# Export an explicit ID-to-split mapping.
clean_df[["mol_id", "split", "scaffold_key"]].to_csv(
    OUTPUT_DIR / "split_ids.csv",
    index=False
)

print("Files written to:", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

Files written to: /content/dual_axis_data
['id_test.csv', 'id_train.csv', 'id_val.csv', 'ood_valid.csv', 'source_invalid_audit.csv', 'split_ids.csv']


In [ ]:

from sklearn.model_selection import train_test_split

SEED = 42

# clean_df was created in Cell 7.
# Each scaffold group must belong entirely to either the ID pool
# or the valid-novel OOD pool.

group_sizes = clean_df.groupby("scaffold_key").size()
group_table = group_sizes.rename("size").reset_index()

rng = np.random.default_rng(SEED)
group_table["tie"] = rng.random(len(group_table))

group_table = group_table.sort_values(
    ["size", "tie"], ascending=[False, True]
)

target_ood = round(0.10 * len(clean_df))
ood_keys = []
ood_count = 0

# Greedily approach the 10% OOD target while keeping
# each scaffold group intact.
for row in group_table.itertuples(index=False):
    if abs(ood_count + row.size - target_ood) < abs(
        ood_count - target_ood
    ):
        ood_keys.append(row.scaffold_key)
        ood_count += row.size

ood_mask = clean_df["scaffold_key"].isin(ood_keys)

ood_df = clean_df.loc[ood_mask].copy()
id_pool = clean_df.loc[~ood_mask].copy()

# Split the ID pool at molecule level.
# These are ID-distribution splits, not scaffold-disjoint splits.
id_train, remainder = train_test_split(
    id_pool,
    test_size=2/9,
    random_state=SEED,
    shuffle=True
)

id_val, id_test = train_test_split(
    remainder,
    test_size=0.5,
    random_state=SEED,
    shuffle=True
)

id_train = id_train.copy()
id_val = id_val.copy()
id_test = id_test.copy()
ood_df = ood_df.copy()

id_train["split"] = "id_train"
id_val["split"] = "id_val"
id_test["split"] = "id_test"
ood_df["split"] = "ood_valid"

final_df = pd.concat(
    [id_train, id_val, id_test, ood_df],
    ignore_index=True
)

print("Split counts:")
print(final_df["split"].value_counts())

print("\nSplit percentages:")
print(
    (final_df["split"].value_counts(normalize=True) * 100).round(2)
)

print("\nScaffold overlap between ID pool and OOD pool:")
print(
    len(
        set(id_pool["scaffold_key"])
        & set(ood_df["scaffold_key"])
    )
)


Split counts:
split
id_train     4938
ood_valid    1474
id_test       706
id_val        705
Name: count, dtype: int64

Split percentages:
split
id_train     63.12
ood_valid    18.84
id_test       9.02
id_val        9.01
Name: proportion, dtype: float64

Scaffold overlap between ID pool and OOD pool:
0


In [ ]:

# 1. Every valid source molecule appears exactly once.
assert len(final_df) == len(clean_df)
assert final_df["mol_id"].nunique() == len(clean_df)

# 2. ID and OOD scaffold groups are disjoint.
id_scaffolds = set(id_pool["scaffold_key"])
ood_scaffolds = set(ood_df["scaffold_key"])

assert id_scaffolds.isdisjoint(ood_scaffolds)

# 3. No canonical molecular structure crosses ID and OOD.
id_structures = set(id_pool["canonical_smiles"])
ood_structures = set(ood_df["canonical_smiles"])

assert id_structures.isdisjoint(ood_structures)

# 4. Report scaffold overlap among ID partitions.
# This is permitted in this design.
for name, part in [
    ("id_train", id_train),
    ("id_val", id_val),
    ("id_test", id_test),
]:
    print(
        name,
        "molecules:", len(part),
        "unique scaffolds:", part["scaffold_key"].nunique()
    )

print("\nAll ID/OOD isolation checks passed.")


id_train molecules: 4938 unique scaffolds: 3353
id_val molecules: 705 unique scaffolds: 595
id_test molecules: 706 unique scaffolds: 599

All ID/OOD isolation checks passed.


In [ ]:

from pathlib import Path

OUTPUT_DIR = Path("/content/dual_axis_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, part in [
    ("id_train", id_train),
    ("id_val", id_val),
    ("id_test", id_test),
    ("ood_valid", ood_df),
]:
    part.drop(columns=["mol"], errors="ignore").to_csv(
        OUTPUT_DIR / f"{name}.csv",
        index=False
    )

final_df[["mol_id", "split", "scaffold_key"]].to_csv(
    OUTPUT_DIR / "split_ids.csv",
    index=False
)

print("Updated files saved to:", OUTPUT_DIR)



Updated files saved to: /content/dual_axis_data


In [ ]:

# Select whole scaffold groups with a target of 782 OOD molecules.
# This subset-sum approach never exceeds the target.

TARGET_OOD = round(0.10 * len(clean_df))
SEED = 42

group_sizes = clean_df.groupby("scaffold_key").size()
groups = group_sizes.reset_index()
groups.columns = ["scaffold_key", "size"]

rng = np.random.default_rng(SEED)
groups = groups.iloc[rng.permutation(len(groups))].reset_index(drop=True)

# reachable[s] tells us whether a subset of groups can total s molecules.
reachable = np.zeros(TARGET_OOD + 1, dtype=bool)
reachable[0] = True

parent_sum = np.full(TARGET_OOD + 1, -1, dtype=int)
parent_group = np.full(TARGET_OOD + 1, -1, dtype=int)

for i, row in groups.iterrows():
    size = int(row["size"])

    if size > TARGET_OOD:
        continue

    for s in range(TARGET_OOD - size, -1, -1):
        if reachable[s] and not reachable[s + size]:
            reachable[s + size] = True
            parent_sum[s + size] = s
            parent_group[s + size] = i

    if reachable[TARGET_OOD]:
        break

# If exactly 782 is impossible, use the largest reachable count below it.
best_total = TARGET_OOD
while best_total > 0 and not reachable[best_total]:
    best_total -= 1

assert best_total > 0

selected_indices = []
s = best_total

while s > 0:
    i = parent_group[s]
    assert i >= 0
    selected_indices.append(i)
    s = parent_sum[s]

ood_keys = set(groups.loc[selected_indices, "scaffold_key"])

ood_mask = clean_df["scaffold_key"].isin(ood_keys)
ood_df = clean_df.loc[ood_mask].copy()
id_pool = clean_df.loc[~ood_mask].copy()

print("Target OOD count:", TARGET_OOD)
print("Actual OOD count:", len(ood_df))
print("ID pool count:", len(id_pool))
print("Scaffold groups assigned to OOD:", len(ood_keys))


Target OOD count: 782
Actual OOD count: 782
ID pool count: 7041
Scaffold groups assigned to OOD: 498


In [ ]:

from sklearn.model_selection import train_test_split

id_train, remainder = train_test_split(
    id_pool,
    test_size=2/9,
    random_state=42,
    shuffle=True
)

id_val, id_test = train_test_split(
    remainder,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

id_train = id_train.copy()
id_val = id_val.copy()
id_test = id_test.copy()
ood_df = ood_df.copy()

id_train["split"] = "id_train"
id_val["split"] = "id_val"
id_test["split"] = "id_test"
ood_df["split"] = "ood_valid"

final_df = pd.concat(
    [id_train, id_val, id_test, ood_df],
    ignore_index=True
)

print(final_df["split"].value_counts())
print(
    (final_df["split"].value_counts(normalize=True) * 100).round(2)
)


split
id_train     5476
id_test       783
id_val        782
ood_valid     782
Name: count, dtype: int64
split
id_train     70.00
id_test      10.01
id_val       10.00
ood_valid    10.00
Name: proportion, dtype: float64


In [ ]:

assert len(final_df) == len(clean_df)
assert final_df["mol_id"].nunique() == len(clean_df)

assert set(id_pool["scaffold_key"]).isdisjoint(
    set(ood_df["scaffold_key"])
)

assert set(id_pool["canonical_smiles"]).isdisjoint(
    set(ood_df["canonical_smiles"])
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, part in [
    ("id_train", id_train),
    ("id_val", id_val),
    ("id_test", id_test),
    ("ood_valid", ood_df),
]:
    part.drop(columns=["mol"], errors="ignore").to_csv(
        OUTPUT_DIR / f"{name}.csv", index=False
    )

final_df[["mol_id", "split", "scaffold_key"]].to_csv(
    OUTPUT_DIR / "split_ids.csv", index=False
)

print("Final isolation checks passed.")
print("Updated files exported to:", OUTPUT_DIR)


Final isolation checks passed.
Updated files exported to: /content/dual_axis_data


In [ ]:

import json
import pandas as pd
import numpy as np
from rdkit import Chem, RDLogger

# Suppress repetitive expected warnings during deliberate corruption.
RDLogger.DisableLog("rdApp.warning")

def graph_to_json(mol):
    """Serialize a graph, including graphs that fail sanitization."""
    return json.dumps({
        "atoms": [
            {
                "atomic_num": atom.GetAtomicNum(),
                "formal_charge": atom.GetFormalCharge(),
                "aromatic": atom.GetIsAromatic(),
                "chiral_tag": str(atom.GetChiralTag()),
            }
            for atom in mol.GetAtoms()
        ],
        "bonds": [
            {
                "begin": bond.GetBeginAtomIdx(),
                "end": bond.GetEndAtomIdx(),
                "type": str(bond.GetBondType()),
                "aromatic": bond.GetIsAromatic(),
                "stereo": str(bond.GetStereo()),
            }
            for bond in mol.GetBonds()
        ],
    })


def create_overvalent_corruption(mol, seed=42):
    """
    Add a single bond involving a saturated, neutral carbon.
    Accept only a candidate that fails RDKit sanitization.
    """
    if mol is None:
        return None

    rng = np.random.default_rng(seed)
    carbon_atoms = [
        atom for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 6
        and atom.GetFormalCharge() == 0
        and not atom.GetIsAromatic()
    ]

    candidates = []

    for atom in carbon_atoms:
        try:
            total_valence = atom.GetTotalValence()
        except Exception:
            continue

        if total_valence < 4:
            continue

        i = atom.GetIdx()

        for other in carbon_atoms:
            j = other.GetIdx()

            if i == j:
                continue

            if mol.GetBondBetweenAtoms(i, j) is not None:
                continue

            candidates.append((i, j))

    if not candidates:
        return None

    # Randomize candidate order reproducibly.
    rng.shuffle(candidates)

    for i, j in candidates:
        rw = Chem.RWMol(mol)

        try:
            rw.AddBond(i, j, Chem.BondType.SINGLE)
            corrupted = rw.GetMol()

            # catchErrors=True returns the first sanitization failure flag.
            failure_flag = Chem.SanitizeMol(
                corrupted,
                catchErrors=True
            )

            if failure_flag != Chem.SanitizeFlags.SANITIZE_NONE:
                return {
                    "graph_json": graph_to_json(corrupted),
                    "corruption_type": "add_bond_overvalent_carbon",
                    "atom_i": i,
                    "atom_j": j,
                    "sanitization_flag": str(failure_flag),
                    "sanitization_failed": True,
                }

        except Exception:
            continue

    return None


# Pilot only: first 100 molecules from ID training data.
pilot_sources = id_train.head(100).copy()

pilot_records = []

for row in pilot_sources.itertuples(index=False):
    result = create_overvalent_corruption(
        row.mol,
        seed=42 + len(pilot_records)
    )

    if result is None:
        pilot_records.append({
            "source_mol_id": row.mol_id,
            "source_split": "id_train",
            "corruption_success": False,
            "graph_json": None,
            "corruption_type": "add_bond_overvalent_carbon",
            "atom_i": None,
            "atom_j": None,
            "sanitization_flag": None,
            "sanitization_failed": False,
        })
    else:
        pilot_records.append({
            "source_mol_id": row.mol_id,
            "source_split": "id_train",
            "corruption_success": True,
            **result,
        })

pilot_df = pd.DataFrame(pilot_records)

RDLogger.EnableLog("rdApp.warning")

print("Attempted:", len(pilot_df))
print("Successful corruptions:", int(pilot_df["corruption_success"].sum()))
print("Failed attempts:", int((~pilot_df["corruption_success"]).sum()))
print(
    "Success rate:",
    round(100 * pilot_df["corruption_success"].mean(), 2),
    "%"
)

pilot_path = OUTPUT_DIR / "corruption_pilot.csv"
pilot_df.to_csv(pilot_path, index=False)

print("Pilot saved to:", pilot_path)


[12:00:37] Explicit valence for atom # 29 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 14 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 4 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 21 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 11 C, 5, is greater than permitted
[12:00:37] Explicit valence for atom # 10 C, 

Attempted: 100
Successful corruptions: 53
Failed attempts: 47
Success rate: 53.0 %
Pilot saved to: /content/dual_axis_data/corruption_pilot.csv


[12:00:38] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:00:38] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:00:38] Explicit valence for atom # 9 C, 5, is greater than permitted
[12:00:38] Explicit valence for atom # 26 C, 5, is greater than permitted


In [ ]:

successful = pilot_df[pilot_df["corruption_success"]].copy()

print("Corruption type counts:")
display(
    pilot_df["corruption_success"]
    .value_counts()
    .rename(index={True: "Success", False: "Failed"})
    .to_frame("count")
)

print("\nSanitization failure flags:")
display(successful["sanitization_flag"].value_counts().to_frame("count"))

print("\nSample successful corruption records:")
display(
    successful[
        [
            "source_mol_id",
            "source_split",
            "corruption_type",
            "atom_i",
            "atom_j",
            "sanitization_flag",
        ]
    ].head(10)
)

assert successful["sanitization_failed"].all()
assert successful["graph_json"].notna().all()
assert successful["source_mol_id"].is_unique

print("\nBasic corruption audit passed.")


Corruption type counts:


,count
corruption_success,
Success,53
Failed,47



Sanitization failure flags:


,count
sanitization_flag,
SANITIZE_PROPERTIES,53



Sample successful corruption records:


,source_mol_id,source_split,corruption_type,atom_i,atom_j,sanitization_flag
3,TOX13469,id_train,add_bond_overvalent_carbon,29.0,10.0,SANITIZE_PROPERTIES
4,TOX1879,id_train,add_bond_overvalent_carbon,0.0,2.0,SANITIZE_PROPERTIES
5,TOX13979,id_train,add_bond_overvalent_carbon,16.0,20.0,SANITIZE_PROPERTIES
6,TOX1641,id_train,add_bond_overvalent_carbon,8.0,5.0,SANITIZE_PROPERTIES
8,TOX26729,id_train,add_bond_overvalent_carbon,0.0,14.0,SANITIZE_PROPERTIES
11,TOX2683,id_train,add_bond_overvalent_carbon,16.0,23.0,SANITIZE_PROPERTIES
12,TOX25602,id_train,add_bond_overvalent_carbon,4.0,8.0,SANITIZE_PROPERTIES
13,TOX29127,id_train,add_bond_overvalent_carbon,4.0,1.0,SANITIZE_PROPERTIES
14,TOX27419,id_train,add_bond_overvalent_carbon,5.0,1.0,SANITIZE_PROPERTIES
16,TOX5642,id_train,add_bond_overvalent_carbon,0.0,5.0,SANITIZE_PROPERTIES



Basic corruption audit passed.


In [ ]:

def create_overvalent_corruption_v2(mol, seed=42):
    """Add a bond to a saturated neutral carbon and verify rejection."""
    if mol is None:
        return None

    rng = np.random.default_rng(seed)

    saturated_carbons = []
    for atom in mol.GetAtoms():
        if (
            atom.GetAtomicNum() == 6
            and atom.GetFormalCharge() == 0
            and not atom.GetIsAromatic()
        ):
            try:
                if atom.GetTotalValence() == 4:
                    saturated_carbons.append(atom.GetIdx())
            except Exception:
                pass

    candidates = []
    for i in saturated_carbons:
        for other in mol.GetAtoms():
            j = other.GetIdx()

            if i == j:
                continue
            if mol.GetBondBetweenAtoms(i, j) is not None:
                continue

            candidates.append((i, j))

    rng.shuffle(candidates)

    for i, j in candidates:
        rw = Chem.RWMol(mol)
        rw.AddBond(i, j, Chem.BondType.SINGLE)
        corrupted = rw.GetMol()

        flag = Chem.SanitizeMol(
            corrupted,
            catchErrors=True
        )

        if flag != Chem.SanitizeFlags.SANITIZE_NONE:
            return {
                "graph_json": graph_to_json(corrupted),
                "corruption_type": "add_bond_overvalent_carbon_v2",
                "atom_i": i,
                "atom_j": j,
                "sanitization_flag": str(flag),
                "sanitization_failed": True,
            }

    return None


pilot_v2_records = []

for index, row in enumerate(id_train.head(100).itertuples(index=False)):
    result = create_overvalent_corruption_v2(
        row.mol, seed=1000 + index
    )

    pilot_v2_records.append({
        "source_mol_id": row.mol_id,
        "source_split": "id_train",
        "corruption_success": result is not None,
        **(result if result is not None else {
            "graph_json": None,
            "corruption_type": "add_bond_overvalent_carbon_v2",
            "atom_i": None,
            "atom_j": None,
            "sanitization_flag": None,
            "sanitization_failed": False,
        })
    })

pilot_v2 = pd.DataFrame(pilot_v2_records)

print("Attempted:", len(pilot_v2))
print("Successful:", int(pilot_v2["corruption_success"].sum()))
print("Failed:", int((~pilot_v2["corruption_success"]).sum()))
print(
    "Success rate:",
    round(100 * pilot_v2["corruption_success"].mean(), 2),
    "%"
)

pilot_v2.to_csv(
    OUTPUT_DIR / "corruption_pilot_v2.csv",
    index=False
)


[12:02:04] Explicit valence for atom # 3 O, 3, is greater than permitted
[12:02:04] Explicit valence for atom # 9 O, 3, is greater than permitted
[12:02:04] Explicit valence for atom # 2 O, 3, is greater than permitted
[12:02:04] Explicit valence for atom # 29 C, 5, is greater than permitted
[12:02:04] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:02:04] Explicit valence for atom # 11 C, 5, is greater than permitted
[12:02:04] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 13
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 13
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 18 19 20 21 23 24 25
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 18 19 20 21 23 24 25
[12:02:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 20 21 25 26 27 30 31
[12:02:04] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 20 

Attempted: 100
Successful: 92
Failed: 8
Success rate: 92.0 %


In [ ]:

successful_v2 = pilot_v2[pilot_v2["corruption_success"]]

assert successful_v2["sanitization_failed"].all()
assert successful_v2["graph_json"].notna().all()
assert pilot_v2["source_mol_id"].is_unique

print("Sanitization flags:")
display(successful_v2["sanitization_flag"].value_counts().to_frame("count"))

print("Successful examples:")
display(
    successful_v2[
        ["source_mol_id", "atom_i", "atom_j", "sanitization_flag"]
    ].head(10)
)


Sanitization flags:


,count
sanitization_flag,
SANITIZE_PROPERTIES,67
SANITIZE_KEKULIZE,25


Successful examples:


,source_mol_id,atom_i,atom_j,sanitization_flag
0,TOX3833,0.0,3.0,SANITIZE_PROPERTIES
1,TOX25296,1.0,9.0,SANITIZE_PROPERTIES
2,TOX21962,4.0,2.0,SANITIZE_PROPERTIES
3,TOX13469,33.0,29.0,SANITIZE_PROPERTIES
4,TOX1879,0.0,2.0,SANITIZE_PROPERTIES
5,TOX13979,16.0,11.0,SANITIZE_PROPERTIES
6,TOX1641,8.0,1.0,SANITIZE_PROPERTIES
7,TOX5091,12.0,10.0,SANITIZE_KEKULIZE
8,TOX26729,41.0,26.0,SANITIZE_KEKULIZE
10,TOX4454,1.0,4.0,SANITIZE_PROPERTIES


In [ ]:

import json
from collections import Counter

def audit_corruption_record(row):
    if not row["corruption_success"]:
        return None

    graph = json.loads(row["graph_json"])
    i = int(row["atom_i"])

    # Sum bond orders incident to the modified carbon.
    bond_order = {
        "SINGLE": 1.0,
        "DOUBLE": 2.0,
        "TRIPLE": 3.0,
        "AROMATIC": 1.5,
    }

    incident_order = 0.0
    incident_bonds = 0

    for bond in graph["bonds"]:
        if i in (bond["begin"], bond["end"]):
            incident_bonds += 1
            incident_order += bond_order.get(bond["type"], 0.0)

    atom = graph["atoms"][i]

    return {
        "source_mol_id": row["source_mol_id"],
        "failure_flag": row["sanitization_flag"],
        "modified_atom": i,
        "atomic_num": atom["atomic_num"],
        "formal_charge": atom["formal_charge"],
        "incident_bonds": incident_bonds,
        "bond_order_sum": incident_order,
        "bond_order_sum_at_least_5": incident_order >= 5.0,
    }

audit_records = [
    audit_corruption_record(row)
    for _, row in pilot_v2.iterrows()
    if row["corruption_success"]
]

audit_df = pd.DataFrame(audit_records)

print("Failure flags:")
display(audit_df["failure_flag"].value_counts().to_frame("count"))

print("\nBond-order sum by failure flag:")
display(
    audit_df.groupby("failure_flag")["bond_order_sum_at_least_5"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nExamples with kekulization failures:")
display(
    audit_df.loc[
        audit_df["failure_flag"] == "SANITIZE_KEKULIZE"
    ].head(10)
)


Failure flags:


,count
failure_flag,
SANITIZE_PROPERTIES,67
SANITIZE_KEKULIZE,25



Bond-order sum by failure flag:


bond_order_sum_at_least_5,False,True
failure_flag,,
SANITIZE_KEKULIZE,25,0
SANITIZE_PROPERTIES,42,25



Examples with kekulization failures:


,source_mol_id,failure_flag,modified_atom,atomic_num,formal_charge,incident_bonds,bond_order_sum,bond_order_sum_at_least_5
7,TOX5091,SANITIZE_KEKULIZE,12,6,0,2,2.0,False
8,TOX26729,SANITIZE_KEKULIZE,41,6,0,3,3.0,False
10,TOX2683,SANITIZE_KEKULIZE,12,6,0,4,4.0,False
15,TOX5642,SANITIZE_KEKULIZE,0,6,0,2,2.0,False
19,TOX26910,SANITIZE_KEKULIZE,14,6,0,3,3.0,False
21,TOX127,SANITIZE_KEKULIZE,5,6,0,3,3.0,False
25,TOX24369,SANITIZE_KEKULIZE,0,6,0,2,2.0,False
33,TOX507,SANITIZE_KEKULIZE,2,6,0,2,2.0,False
38,TOX18686,SANITIZE_KEKULIZE,0,6,0,2,2.0,False
40,TOX6302,SANITIZE_KEKULIZE,0,6,0,2,2.0,False


In [ ]:
import json
import random
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdchem

def create_overvalent_corruption_v3(mol, seed=42):
    """
    Add a single bond between two non-aromatic, neutral carbon atoms.
    Retain only examples where the chosen carbon becomes over-valent
    and RDKit sanitization fails.
    """
    if mol is None:
        return None

    rng = random.Random(seed)
    candidates = [
        atom.GetIdx()
        for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 6
        and atom.GetFormalCharge() == 0
        and not atom.GetIsAromatic()
        and atom.GetTotalValence() == 4
    ]

    rng.shuffle(candidates)

    for i in candidates:
        possible_j = [
            atom.GetIdx()
            for atom in mol.GetAtoms()
            if atom.GetIdx() != i
            and atom.GetAtomicNum() == 6
            and atom.GetFormalCharge() == 0
            and not atom.GetIsAromatic()
            and mol.GetBondBetweenAtoms(i, atom.GetIdx()) is None
        ]
        rng.shuffle(possible_j)

        for j in possible_j:
            corrupted = Chem.RWMol(mol)
            corrupted.AddBond(i, j, rdchem.BondType.SINGLE)
            corrupted = corrupted.GetMol()

            # Compute valence properties without enforcing validity yet.
            corrupted.UpdatePropertyCache(strict=False)
            modified_atom = corrupted.GetAtomWithIdx(i)
            explicit_valence = modified_atom.GetValence(
    Chem.rdchem.ValenceType.EXPLICIT
)

            # Reject examples that do not show the intended over-valence.
            if explicit_valence <= 4:
                continue

            # Capture the first sanitization failure.
            failure = Chem.SanitizeMol(
                corrupted,
                catchErrors=True
            )

            if failure == Chem.SanitizeFlags.SANITIZE_NONE:
                continue

            return {
                "graph_json": graph_to_json(corrupted),
                "atom_i": i,
                "atom_j": j,
                "explicit_valence_i": explicit_valence,
                "sanitization_flag": str(failure),
                "corruption_type": "add_bond_overvalent_carbon_v3",
            }

    return None


# Small pilot first — do not generate the full dataset yet.
pilot_rows = []

for index, row in enumerate(id_train.head(100).itertuples(index=False)):
    result = create_overvalent_corruption_v3(
        row.mol,
        seed=1000 + index
    )

    pilot_rows.append({
        "source_mol_id": row.mol_id,
        "source_split": "id_train",
        "corruption_success": result is not None,
        **(result or {}),
    })

pilot_v3 = pd.DataFrame(pilot_rows)
pilot_v3.to_csv(
    OUTPUT_DIR / "corruption_pilot_v3.csv",
    index=False
)

print("Attempted:", len(pilot_v3))
print("Successful:", int(pilot_v3["corruption_success"].sum()))
print(
    "Success rate:",
    f"{pilot_v3['corruption_success'].mean() * 100:.1f}%"
)

print("\nSanitization flags among successful examples:")
display(
    pilot_v3.loc[pilot_v3["corruption_success"],
                 "sanitization_flag"]
    .value_counts()
    .to_frame("count")
)

Attempted: 100
Successful: 53
Success rate: 53.0%

Sanitization flags among successful examples:


[12:07:23] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 22 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 23 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 23 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:07:23] Explicit valence for atom # 4 C, 5,

,count
sanitization_flag,
SANITIZE_PROPERTIES,53


In [ ]:
import pandas as pd
from rdkit import Chem

def generate_corruptions_for_split(split_df, split_name):
    records = []

    for index, row in enumerate(split_df.itertuples(index=False)):
        result = create_overvalent_corruption_v3(
            row.mol,
            seed=42000 + index
        )

        record = {
            "source_mol_id": row.mol_id,
            "source_split": split_name,
            "corruption_success": result is not None,
        }

        if result is not None:
            record.update(result)

        records.append(record)

    return pd.DataFrame(records)


corrupt_train = generate_corruptions_for_split(
    id_train, "id_train"
)
corrupt_val = generate_corruptions_for_split(
    id_val, "id_val"
)
corrupt_test = generate_corruptions_for_split(
    id_test, "id_test"
)

# Keep successful corruptions for the main invalid-graph datasets.
invalid_train = corrupt_train.loc[
    corrupt_train["corruption_success"]
].copy()

invalid_val = corrupt_val.loc[
    corrupt_val["corruption_success"]
].copy()

invalid_test = corrupt_test.loc[
    corrupt_test["corruption_success"]
].copy()

# Export full audit tables, including failed attempts.
corrupt_train.to_csv(
    OUTPUT_DIR / "corruption_audit_train.csv", index=False
)
corrupt_val.to_csv(
    OUTPUT_DIR / "corruption_audit_val.csv", index=False
)
corrupt_test.to_csv(
    OUTPUT_DIR / "corruption_audit_test.csv", index=False
)

invalid_train.to_csv(
    OUTPUT_DIR / "invalid_graphs_train.csv", index=False
)
invalid_val.to_csv(
    OUTPUT_DIR / "invalid_graphs_val.csv", index=False
)
invalid_test.to_csv(
    OUTPUT_DIR / "invalid_graphs_test.csv", index=False
)

summary = pd.DataFrame([
    {
        "split": name,
        "attempted": len(audit),
        "successful": int(audit["corruption_success"].sum()),
        "failed": int((~audit["corruption_success"]).sum()),
        "success_rate_pct": round(
            audit["corruption_success"].mean() * 100, 2
        ),
    }
    for name, audit in [
        ("id_train", corrupt_train),
        ("id_val", corrupt_val),
        ("id_test", corrupt_test),
    ]
])

display(summary)

[12:08:15] Explicit valence for atom # 26 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 12 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 29 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 25 C, 5, is greater than permitted
[12:08:15] Explicit valence for atom # 10 C, 5

,split,attempted,successful,failed,success_rate_pct
0,id_train,5476,3237,2239,59.11
1,id_val,782,481,301,61.51
2,id_test,783,477,306,60.92


In [ ]:
import json
import pandas as pd

audit_tables = {
    "id_train": corrupt_train,
    "id_val": corrupt_val,
    "id_test": corrupt_test,
}

invalid_tables = {
    "id_train": invalid_train,
    "id_val": invalid_val,
    "id_test": invalid_test,
}

validation_results = []

for split_name, audit in audit_tables.items():
    invalid = invalid_tables[split_name]
    successful = audit[audit["corruption_success"]].copy()

    # Every retained example should have a non-empty JSON graph.
    json_ok = successful["graph_json"].apply(
        lambda value: isinstance(value, str)
        and len(value) > 0
        and isinstance(json.loads(value), dict)
    ).all()

    # A source molecule should not occur twice within one split.
    unique_sources = (
        successful["source_mol_id"].nunique()
        == len(successful)
    )

    # Verify the intended corruption label.
    correct_type = successful["corruption_type"].eq(
        "add_bond_overvalent_carbon_v3"
    ).all()

    # Verify each successful graph contains atom and bond lists.
    graph_structure_ok = successful["graph_json"].apply(
        lambda value: (
            isinstance((graph := json.loads(value)).get("atoms"), list)
            and isinstance(graph.get("bonds"), list)
        )
    ).all()

    validation_results.append({
        "split": split_name,
        "successful_examples": len(successful),
        "graph_json_valid": bool(json_ok),
        "graph_has_atoms_and_bonds": bool(graph_structure_ok),
        "unique_source_ids": bool(unique_sources),
        "correct_corruption_type": bool(correct_type),
    })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

print(
    "All checks passed:",
    bool(validation_df.iloc[:, 2:].to_numpy().all())
)

,split,successful_examples,graph_json_valid,graph_has_atoms_and_bonds,unique_source_ids,correct_corruption_type
0,id_train,3237,True,True,True,True
1,id_val,481,True,True,True,True
2,id_test,477,True,True,True,True


All checks passed: True


In [ ]:
import json
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdchem

BOND_TYPES = {
    "SINGLE": rdchem.BondType.SINGLE,
    "DOUBLE": rdchem.BondType.DOUBLE,
    "TRIPLE": rdchem.BondType.TRIPLE,
    "AROMATIC": rdchem.BondType.AROMATIC,
}

def reconstruct_graph(graph_json):
    data = json.loads(graph_json)
    rw = Chem.RWMol()

    for item in data["atoms"]:
        atom = Chem.Atom(int(item["atomic_num"]))
        atom.SetFormalCharge(int(item["formal_charge"]))
        atom.SetIsAromatic(bool(item["aromatic"]))
        rw.AddAtom(atom)

    for item in data["bonds"]:
        bond_name = str(item["type"]).split(".")[-1]
        bond_type = BOND_TYPES[bond_name]

        rw.AddBond(
            int(item["begin"]),
            int(item["end"]),
            bond_type
        )

        bond = rw.GetBondBetweenAtoms(
            int(item["begin"]),
            int(item["end"])
        )
        bond.SetIsAromatic(bool(item["aromatic"]))

    return rw.GetMol()


validation_records = []

for split_name, invalid_df in [
    ("train", invalid_train),
    ("val", invalid_val),
    ("test", invalid_test),
]:
    for row in invalid_df.itertuples(index=False):
        try:
            mol = reconstruct_graph(row.graph_json)
            result = Chem.SanitizeMol(mol, catchErrors=True)

            validation_records.append({
                "split": split_name,
                "source_mol_id": row.source_mol_id,
                "sanitization_failed": (
                    result != Chem.SanitizeFlags.SANITIZE_NONE
                ),
                "failure_flag": str(result),
                "error": None,
            })

        except Exception as exc:
            validation_records.append({
                "split": split_name,
                "source_mol_id": row.source_mol_id,
                "sanitization_failed": None,
                "failure_flag": None,
                "error": str(exc),
            })

rdkit_validation = pd.DataFrame(validation_records)

display(
    rdkit_validation.groupby("split").agg(
        total=("source_mol_id", "size"),
        sanitization_failed=("sanitization_failed", "sum"),
        reconstruction_errors=("error", lambda s: s.notna().sum()),
    )
)

print(
    "All reconstructed graphs failed sanitization:",
    bool(
        rdkit_validation["sanitization_failed"].notna().all()
        and rdkit_validation["sanitization_failed"].all()
        and rdkit_validation["error"].isna().all()
    )
)

rdkit_validation.to_csv(
    OUTPUT_DIR / "rdkit_reconstruction_validation.csv",
    index=False
)

[12:10:52] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 42 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 25 C, 5, is greater than permitted
[12:10:52] Can't kekulize mol.  Unkekulized atoms: 1 2 16
[12:10:52] Can't kekulize mol.  Unkekulized atoms: 1 2 16
[12:10:52] Explicit valence for atom # 3 C, 5, is greater than permitted
[12:10:52] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:1

,total,sanitization_failed,reconstruction_errors
split,,,
test,477,422,0
train,3237,2886,0
val,481,419,0


All reconstructed graphs failed sanitization: False


In [ ]:
passed_after_reconstruction = rdkit_validation[
    rdkit_validation["sanitization_failed"].eq(False)
].copy()

print("Graphs that unexpectedly passed:",
      len(passed_after_reconstruction))

display(
    passed_after_reconstruction[
        ["split", "source_mol_id", "failure_flag"]
    ].head(20)
)


Graphs that unexpectedly passed: 468


,split,source_mol_id,failure_flag
0,train,TOX13469,SANITIZE_NONE
2,train,TOX13979,SANITIZE_NONE
6,train,TOX25602,SANITIZE_NONE
15,train,TOX3004,SANITIZE_NONE
37,train,TOX27795,SANITIZE_NONE
67,train,TOX26647,SANITIZE_NONE
72,train,TOX28923,SANITIZE_NONE
104,train,TOX28717,SANITIZE_NONE
117,train,TOX27052,SANITIZE_NONE
125,train,TOX25479,SANITIZE_NONE


In [ ]:
import json
import pandas as pd
from rdkit import Chem

# Pick a few examples that unexpectedly passed reconstruction.
sample_ids = passed_after_reconstruction[
    "source_mol_id"
].head(5).tolist()

comparison = []

for source_id in sample_ids:
    original_row = invalid_train.loc[
        invalid_train["source_mol_id"] == source_id
    ].iloc[0]

    graph_data = json.loads(original_row["graph_json"])
    reconstructed = reconstruct_graph(original_row["graph_json"])

    i = int(original_row["atom_i"])
    j = int(original_row["atom_j"])

    # Inspect the target atom and added bond.
    target_atom = reconstructed.GetAtomWithIdx(i)
    added_bond = reconstructed.GetBondBetweenAtoms(i, j)

    reconstructed.UpdatePropertyCache(strict=False)

    comparison.append({
        "source_mol_id": source_id,
        "recorded_valence_i": original_row["explicit_valence_i"],
        "reconstructed_explicit_valence_i": (
            target_atom.GetValence(
                Chem.rdchem.ValenceType.EXPLICIT
            )
        ),
        "reconstructed_explicit_valence_i": (
    target_atom.GetValence(
        Chem.rdchem.ValenceType.EXPLICIT
    )
),
        "added_bond_exists": added_bond is not None,
        "added_bond_type": (
            str(added_bond.GetBondType())
            if added_bond is not None else None
        ),
        "atom_i_aromatic": target_atom.GetIsAromatic(),
        "atom_j_aromatic": (
            reconstructed.GetAtomWithIdx(j).GetIsAromatic()
        ),
    })

display(pd.DataFrame(comparison))

,source_mol_id,recorded_valence_i,reconstructed_explicit_valence_i,added_bond_exists,added_bond_type,atom_i_aromatic,atom_j_aromatic
0,TOX13469,5.0,4,True,SINGLE,False,False
1,TOX13979,5.0,4,True,SINGLE,False,False
2,TOX25602,5.0,4,True,SINGLE,False,False
3,TOX3004,5.0,4,True,SINGLE,False,False
4,TOX27795,5.0,4,True,SINGLE,False,False


In [ ]:
import json
from rdkit import Chem

def graph_to_json(mol):
    atoms = []

    for atom in mol.GetAtoms():
        atoms.append({
            "atomic_num": atom.GetAtomicNum(),
            "formal_charge": atom.GetFormalCharge(),
            "aromatic": atom.GetIsAromatic(),
            "chiral_tag": str(atom.GetChiralTag()),
            "explicit_hs": atom.GetNumExplicitHs(),
            "no_implicit": atom.GetNoImplicit(),
        })

    bonds = []

    for bond in mol.GetBonds():
        bonds.append({
            "begin": bond.GetBeginAtomIdx(),
            "end": bond.GetEndAtomIdx(),
            "type": str(bond.GetBondType()),
            "aromatic": bond.GetIsAromatic(),
            "stereo": str(bond.GetStereo()),
        })

    return json.dumps({
        "atoms": atoms,
        "bonds": bonds,
    })

print("Updated graph_to_json successfully.")

Updated graph_to_json successfully.


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdchem

BOND_TYPES = {
    "SINGLE": rdchem.BondType.SINGLE,
    "DOUBLE": rdchem.BondType.DOUBLE,
    "TRIPLE": rdchem.BondType.TRIPLE,
    "AROMATIC": rdchem.BondType.AROMATIC,
}

def reconstruct_graph(graph_json):
    data = json.loads(graph_json)
    rw = Chem.RWMol()

    for item in data["atoms"]:
        atom = Chem.Atom(int(item["atomic_num"]))
        atom.SetFormalCharge(int(item["formal_charge"]))
        atom.SetIsAromatic(bool(item["aromatic"]))
        atom.SetNumExplicitHs(int(item.get("explicit_hs", 0)))
        atom.SetNoImplicit(bool(item.get("no_implicit", False)))
        rw.AddAtom(atom)

    for item in data["bonds"]:
        bond_name = str(item["type"]).split(".")[-1]
        bond_type = BOND_TYPES[bond_name]

        rw.AddBond(
            int(item["begin"]),
            int(item["end"]),
            bond_type
        )

        bond = rw.GetBondBetweenAtoms(
            int(item["begin"]),
            int(item["end"])
        )
        bond.SetIsAromatic(bool(item["aromatic"]))

    return rw.GetMol()

print("Updated reconstruct_graph successfully.")

Updated reconstruct_graph successfully.


In [ ]:
# Test one source molecule using your existing v3 function.
test_row = next(id_train.itertuples(index=False))

result = create_overvalent_corruption_v3(
    test_row.mol,
    seed=42000
)

if result is None:
    print("No corruption found for this molecule. Try another source.")
else:
    rebuilt = reconstruct_graph(result["graph_json"])
    rebuilt.UpdatePropertyCache(strict=False)

    atom_i = result["atom_i"]

    print("Recorded explicit valence:",
          result["explicit_valence_i"])
    print("Reconstructed explicit valence:",
          rebuilt.GetAtomWithIdx(atom_i).GetValence(
              Chem.rdchem.ValenceType.EXPLICIT
          ))
    print("Sanitization result:",
          Chem.SanitizeMol(rebuilt, catchErrors=True))

No corruption found for this molecule. Try another source.


In [ ]:
from rdkit import Chem

test_result = None
test_source_id = None

for idx, row in enumerate(id_train.itertuples(index=False)):
    result = create_overvalent_corruption_v3(
        row.mol,
        seed=42000 + idx
    )

    if result is not None:
        test_result = result
        test_source_id = row.mol_id
        break

if test_result is None:
    print("No suitable corruption found in id_train.")
else:
    rebuilt = reconstruct_graph(test_result["graph_json"])
    rebuilt.UpdatePropertyCache(strict=False)

    atom_i = test_result["atom_i"]

    print("Source molecule:", test_source_id)
    print("Recorded explicit valence:",
          test_result["explicit_valence_i"])
    print("Reconstructed explicit valence:",
          rebuilt.GetAtomWithIdx(atom_i).GetValence(
              Chem.rdchem.ValenceType.EXPLICIT
          ))

    sanitize_result = Chem.SanitizeMol(
        rebuilt,
        catchErrors=True
    )
    print("Sanitization result:", sanitize_result)

Source molecule: TOX13469
Recorded explicit valence: 5
Reconstructed explicit valence: 5
Sanitization result: SANITIZE_PROPERTIES


[12:17:25] Explicit valence for atom # 26 C, 5, is greater than permitted
[12:17:25] Explicit valence for atom # 26 C, 5, is greater than permitted


In [ ]:
import pandas as pd

def generate_corruptions_for_split(split_df, split_name):
    records = []

    for index, row in enumerate(split_df.itertuples(index=False)):
        result = create_overvalent_corruption_v3(
            row.mol,
            seed=42000 + index
        )

        record = {
            "source_mol_id": row.mol_id,
            "source_split": split_name,
            "corruption_success": result is not None,
        }

        if result is not None:
            record.update(result)

        records.append(record)

    return pd.DataFrame(records)


# Regenerate the datasets with the updated serializer.
corrupt_train = generate_corruptions_for_split(id_train, "id_train")
corrupt_val = generate_corruptions_for_split(id_val, "id_val")
corrupt_test = generate_corruptions_for_split(id_test, "id_test")

invalid_train = corrupt_train[corrupt_train["corruption_success"]].copy()
invalid_val = corrupt_val[corrupt_val["corruption_success"]].copy()
invalid_test = corrupt_test[corrupt_test["corruption_success"]].copy()

# Save audit tables, including unsuccessful attempts.
corrupt_train.to_csv(OUTPUT_DIR / "corruption_audit_train.csv", index=False)
corrupt_val.to_csv(OUTPUT_DIR / "corruption_audit_val.csv", index=False)
corrupt_test.to_csv(OUTPUT_DIR / "corruption_audit_test.csv", index=False)

# Save successful invalid graphs.
invalid_train.to_csv(OUTPUT_DIR / "invalid_graphs_train.csv", index=False)
invalid_val.to_csv(OUTPUT_DIR / "invalid_graphs_val.csv", index=False)
invalid_test.to_csv(OUTPUT_DIR / "invalid_graphs_test.csv", index=False)

summary = pd.DataFrame([
    {
        "split": name,
        "attempted": len(audit),
        "successful": int(audit["corruption_success"].sum()),
        "failed": int((~audit["corruption_success"]).sum()),
        "success_rate_pct": round(
            audit["corruption_success"].mean() * 100, 2
        ),
    }
    for name, audit in [
        ("train", corrupt_train),
        ("val", corrupt_val),
        ("test", corrupt_test),
    ]
])

display(summary)

[12:17:49] Explicit valence for atom # 26 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 12 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 29 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 25 C, 5, is greater than permitted
[12:17:49] Explicit valence for atom # 10 C, 5

,split,attempted,successful,failed,success_rate_pct
0,train,5476,3237,2239,59.11
1,val,782,481,301,61.51
2,test,783,477,306,60.92


In [ ]:
import json
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdchem

# Map serialized bond types to RDKit bond types
BOND_TYPES = {
    "SINGLE": rdchem.BondType.SINGLE,
    "DOUBLE": rdchem.BondType.DOUBLE,
    "TRIPLE": rdchem.BondType.TRIPLE,
    "AROMATIC": rdchem.BondType.AROMATIC,
}


def reconstruct_graph(graph_json):
    """Reconstruct an RDKit molecule from serialized graph JSON."""
    data = json.loads(graph_json)
    rw = Chem.RWMol()

    # Reconstruct atoms and preserve hydrogen-related properties
    for item in data["atoms"]:
        atom = Chem.Atom(int(item["atomic_num"]))
        atom.SetFormalCharge(int(item["formal_charge"]))
        atom.SetIsAromatic(bool(item["aromatic"]))
        atom.SetNumExplicitHs(int(item.get("explicit_hs", 0)))
        atom.SetNoImplicit(bool(item.get("no_implicit", False)))
        rw.AddAtom(atom)

    # Reconstruct bonds
    for item in data["bonds"]:
        bond_name = str(item["type"]).split(".")[-1]
        bond_type = BOND_TYPES[bond_name]

        rw.AddBond(
            int(item["begin"]),
            int(item["end"]),
            bond_type
        )

        bond = rw.GetBondBetweenAtoms(
            int(item["begin"]),
            int(item["end"])
        )
        bond.SetIsAromatic(bool(item["aromatic"]))

    return rw.GetMol()


# Validate every saved corrupted graph
validation_records = []

for split_name, invalid_df in [
    ("train", invalid_train),
    ("val", invalid_val),
    ("test", invalid_test),
]:
    for row in invalid_df.itertuples(index=False):
        try:
            mol = reconstruct_graph(row.graph_json)

            # A nonzero sanitization flag indicates sanitization failure
            result = Chem.SanitizeMol(
                mol,
                catchErrors=True
            )

            validation_records.append({
                "split": split_name,
                "source_mol_id": row.source_mol_id,
                "sanitization_failed": (
                    result != Chem.SanitizeFlags.SANITIZE_NONE
                ),
                "failure_flag": str(result),
                "error": None,
            })

        except Exception as exc:
            validation_records.append({
                "split": split_name,
                "source_mol_id": row.source_mol_id,
                "sanitization_failed": None,
                "failure_flag": None,
                "error": str(exc),
            })


# Build validation report
rdkit_validation = pd.DataFrame(validation_records)

summary = (
    rdkit_validation
    .groupby("split")
    .agg(
        total=("source_mol_id", "size"),
        sanitization_failed=("sanitization_failed", "sum"),
        reconstruction_errors=("error", lambda s: s.notna().sum()),
    )
    .reset_index()
)

summary["sanitization_passed"] = (
    summary["total"] - summary["sanitization_failed"]
)

summary["failure_rate_pct"] = (
    summary["sanitization_failed"] / summary["total"] * 100
).round(2)

print("RDKit Reconstruction Validation")
display(summary)

all_failed = (
    len(rdkit_validation) > 0
    and rdkit_validation["sanitization_failed"].notna().all()
    and rdkit_validation["sanitization_failed"].all()
    and rdkit_validation["error"].isna().all()
)

print("All reconstructed graphs failed sanitization:", all_failed)

# Save detailed and summary reports
rdkit_validation.to_csv(
    OUTPUT_DIR / "rdkit_reconstruction_validation.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "rdkit_validation_summary.csv",
    index=False
)

print("\nSaved reports:")
print(OUTPUT_DIR / "rdkit_reconstruction_validation.csv")
print(OUTPUT_DIR / "rdkit_validation_summary.csv")

[12:19:00] Explicit valence for atom # 26 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 12 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 8 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 29 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 16 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 1 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 5 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 6 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 2 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 25 C, 5, is greater than permitted
[12:19:00] Explicit valence for atom # 10 C, 5

RDKit Reconstruction Validation


,split,total,sanitization_failed,reconstruction_errors,sanitization_passed,failure_rate_pct
0,test,477,477,0,0,100.0
1,train,3237,3237,0,0,100.0
2,val,481,481,0,0,100.0


All reconstructed graphs failed sanitization: True

Saved reports:
/content/dual_axis_data/rdkit_reconstruction_validation.csv
/content/dual_axis_data/rdkit_validation_summary.csv


In [ ]:
import json
from datetime import datetime, timezone

manifest = {
    "project": "Dual-Axis Out-of-Distribution Detection",
    "dataset": "Tox21",
    "random_seed": 42,
    "split_strategy": {
        "id_train": "70% of the complete valid dataset",
        "id_val": "10% of the complete valid dataset",
        "id_test": "10% of the complete valid dataset",
        "ood_valid": "10% of the complete valid dataset",
        "ood_definition": (
            "Scaffold-held-out; acyclic molecules use canonical "
            "SMILES as a fallback scaffold key"
        ),
    },
    "valid_molecule_counts": {
        "id_train": len(id_train),
        "id_val": len(id_val),
        "id_test": len(id_test),
        "ood_valid": len(ood_df),
    },
    "invalid_graph_generation": {
        "method": "add_bond_overvalent_carbon_v3",
        "train": len(invalid_train),
        "val": len(invalid_val),
        "test": len(invalid_test),
        "total": (
            len(invalid_train)
            + len(invalid_val)
            + len(invalid_test)
        ),
    },
    "reconstruction_validation": {
        "total_graphs": len(rdkit_validation),
        "sanitization_failures": int(
            rdkit_validation["sanitization_failed"].sum()
        ),
        "reconstruction_errors": int(
            rdkit_validation["error"].notna().sum()
        ),
        "all_graphs_failed_sanitization": bool(all_failed),
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = OUTPUT_DIR / "dataset_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print(f"\nManifest saved to: {manifest_path}")

{
  "project": "Dual-Axis Out-of-Distribution Detection",
  "dataset": "Tox21",
  "random_seed": 42,
  "split_strategy": {
    "id_train": "70% of the complete valid dataset",
    "id_val": "10% of the complete valid dataset",
    "id_test": "10% of the complete valid dataset",
    "ood_valid": "10% of the complete valid dataset",
    "ood_definition": "Scaffold-held-out; acyclic molecules use canonical SMILES as a fallback scaffold key"
  },
  "valid_molecule_counts": {
    "id_train": 5476,
    "id_val": 782,
    "id_test": 783,
    "ood_valid": 782
  },
  "invalid_graph_generation": {
    "method": "add_bond_overvalent_carbon_v3",
    "train": 3237,
    "val": 481,
    "test": 477,
    "total": 4195
  },
  "reconstruction_validation": {
    "total_graphs": 4195,
    "sanitization_failures": 4195,
    "reconstruction_errors": 0,
    "all_graphs_failed_sanitization": true
  },
  "created_at_utc": "2026-10-11T12:19:37.135636+00:00"
}

Manifest saved to: /content/dual_axis_data/dataset_

In [ ]:
import shutil
from google.colab import files

# Compress the complete dataset folder
zip_path = shutil.make_archive(
    "/content/dual_axis_data_backup",
    "zip",
    "/content/dual_axis_data"
)

print("ZIP created:", zip_path)

# Download it to your computer
files.download(zip_path)

ZIP created: /content/dual_axis_data_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>